# Dataset Mention Extraction with `ai4data`

This notebook demonstrates how to use the `ai4data` library to extract dataset mentions from text and documents. 

The library features a two-pass extraction pipeline:
1. **Pass 1 (Token Classification & Relation Extraction)**: Extracts proper name spans (`named_data`, `descriptive_data`, `vague_data`) and links them to factual metadata (such as acronyms, producers, and years) via relations.
2. **Pass 2 (Native Sentence Classification)**: Runs sentence-level DeBERTa classification on the extracted contexts to predict the `usage_context` (`primary`, `supporting`, `background`) and `typology_tag` (`survey`, `census`, `database`, etc.). This bypasses token overlapping constraints and yields highly accurate predictions.

## Setup

In [ ]:
import os
import sys
import json

# Ensure the src directory of the repository is in python path
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from ai4data import extract_from_text, extract_from_document, DatasetExtractor

## 1. Extracting Mentions from Text

We can extract mentions directly from a text block. The `extract_from_text` helper function loads the default model and adapter, processes the text, and returns the structured results.

In [ ]:
text = (
    "We conduct our main analysis using microdata from the 2018 Nigeria General "
    "Household Survey (GHS), which was conducted by the National Bureau of Statistics "
    "(NBS) in collaboration with the World Bank. To support the main findings, we "
    "also consult the World Development Indicators database."
)

print("--- Input Text ---")
print(text)

In [ ]:
result = extract_from_text(
    text,
    include_confidence=True,
    adapter_id="ai4data/datause-extraction"
)

In [ ]:
print("--- Extracted Datasets ---\n")
for ds in result.get("datasets", []):
    name = ds["mention_name"]["text"]
    conf = ds["mention_name"]["confidence"]
    typology = ds["typology_tag"]["text"]
    typology_conf = ds["typology_tag"]["confidence"]
    usage = ds["usage_context"]["text"]
    usage_conf = ds["usage_context"]["confidence"]
    
    print(f"Mention Name:  {name} (conf: {conf:.3f})")
    print(f"  Typology Tag: {typology} (conf: {typology_conf:.3f})")
    print(f"  Usage Context:{usage} (conf: {usage_conf:.3f})")
    print(f"  Is Used:      {ds['is_used']['text']} (conf: {ds['is_used']['confidence']:.3f})")
    print(f"  Acronym:      {ds['acronym']['text'] or 'None'}")
    print(f"  Producer:     {ds['producer']['text'] or 'None'}")
    print(f"  Year:         {ds['reference_year']['text'] or 'None'}")
    print("-" * 50)

## 2. Extracting Mentions from a PDF Document

The `extract_from_document` function allows processing full PDF documents. It parses the document page-by-page, chunking where necessary, applying heuristics filters, and performing pre-filtering to skip non-English pages or pages containing no dataset references.

In [ ]:
# In this demo we will extract the first few pages of the Ghana mining PDF if available
pdf_path = "/Users/rafaelmacalaba/WBG/monitoring_of_datause/finetuning/cached_extractions/new_beginnings/ghana_gold_mining.pdf"

if os.path.exists(pdf_path):
    print(f"Found PDF: {pdf_path}. Running extraction on pages 1 to 5...")
    # Note: pages parameter is 0-indexed
    doc_results = extract_from_document(
        pdf_path,
        include_confidence=True,
        pages=[0, 1, 2, 3, 4]  # Extract first 5 pages
    )
    
    # Inspect pages with extracted datasets
    for page_res in doc_results:
        page_num = page_res["page_number"]
        datasets = page_res.get("datasets", [])
        if datasets:
            print(f"\n=== Page {page_num} ({len(datasets)} dataset(s) found) ===")
            for ds in datasets:
                print(f"- Mention: {ds['mention_name']['text']}")
                print(f"  Typology: {ds['typology_tag']['text']} (conf: {ds['typology_tag']['confidence']:.3f})")
                print(f"  Usage:    {ds['usage_context']['text']} (conf: {ds['usage_context']['confidence']:.3f})")
else:
    print(f"PDF file not found at {pdf_path}. Please check path to run this section.")

## 3. Using the `DatasetExtractor` Object directly

For advanced uses or to run extraction within a larger pipeline (e.g. holding the model weights in memory), instantiate a `DatasetExtractor` directly:

In [ ]:
extractor = DatasetExtractor(adapter_id="ai4data/datause-extraction")

sample = "This study utilizes data from the UNHCR PRIMES registration system."
extraction = extractor.extract_from_text(sample, include_confidence=True)

print(json.dumps(extraction["datasets"], indent=2))